# APIM ❤️ FinOps

## Product Framework lab
![flow](../../images/product-framework.gif)

This playground leverages the [FinOps Framework](https://www.finops.org/framework/) and Azure API Management to control AI costs. It uses the [token limit](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-token-limit-policy) policy for each [product](https://learn.microsoft.com/en-us/azure/api-management/api-management-howto-add-products?tabs=azure-portal&pivots=interactive) and integrates [Azure Monitor alerts](https://learn.microsoft.com/en-us/azure/azure-monitor/alerts/alerts-overview) with [Logic Apps](https://learn.microsoft.com/en-us/azure/azure-monitor/alerts/alerts-logic-apps?tabs=send-email) to automatically disable APIM [subscriptions](https://learn.microsoft.com/en-us/azure/api-management/api-management-subscriptions) that exceed cost quotas.

For this sample, each APIM product (`gpt5`, `gpt5mini`, and `unlimited`) has a monthly budget of **£100** (GBP).

### Result
![result](result.png)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the OpenAI model and version according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [12]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "westeurope"

aiservices_config = [{"name": "foundry1", "location": "swedencentral"}]

models_config = [ { "name": "gpt-5-mini", "publisher": "OpenAI", "version": "2025-08-07", "sku": "GlobalStandard", "capacity": 150, "inputTokensMeterSku": "gpt 5 mini Inp glbl", "outputTokensMeterSku": "gpt 5 mini Outp glbl" }, 
                { "name": "gpt-5", "publisher": "OpenAI", "version": "2025-08-07", "sku": "GlobalStandard", "capacity": 150, "inputTokensMeterSku": "gpt 5 Inp glbl", "outputTokensMeterSku": "gpt 5 Outp glbl" } ]

apim_sku = 'Basicv2'
apim_products_config = [{"name": "gpt5", "displayName": "GPT5 Product", "tpm": 800, "tokenQuota": 500000, "tokenQuotaPeriod": "Monthly", "costQuota": 100 },
                    {"name": "gpt5mini", "displayName": "GPT5 Mini Product", "tpm": 2000, "tokenQuota": 2000000, "tokenQuotaPeriod": "Monthly", "costQuota": 100}, 
                    {"name": "unlimited", "displayName": "Unlimited Product", "tpm": 10000, "tokenQuota": 50000000, "tokenQuotaPeriod": "Monthly", "costQuota": 100}]
apim_users_config = [ ]
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1", "product": "gpt5" },
                    {"name": "subscription2", "displayName": "Subscription 2", "product": "gpt5mini" },
                    {"name": "subscription3", "displayName": "Subscription 3", "product": "unlimited" } ]

inference_api_path = "inference"  # path to the inference API in the APIM service
inference_api_type = "AzureOpenAI"  # options: AzureOpenAI, AzureAI, OpenAI, PassThrough
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

currency_code = 'GBP'

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 10:52:59.618875 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [13]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

output = utils.run("az ad signed-in-user show", "Retrieved az ad signed-in-user", "Failed to get az ad signed-in-user")
if output.success and output.json_data:
    current_user_object_id = output.json_data['id']

    

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 10:53:05.109780 :2s]
👉🏽 Current user: admin@MngEnvMCAP734518.onmicrosoft.com
👉🏽 Tenant ID: cdfe81b5-821e-4f07-9ea7-516efc8497e4
👉🏽 Subscription ID: 3d2c527a-481d-4e13-b3a1-637924b33343
⚙️ Running: az ad signed-in-user show 
✅ Retrieved az ad signed-in-user ⌚ 10:53:08.916239 :3s]


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations. 

⚠️ Retry this step if you get deployment error: `workspace not active` 

In [14]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimUsersConfig": { "value": apim_users_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "apimProductsConfig": { "value": apim_products_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

⚙️ Running: az group show --name lab-product-framework 
👉🏽 Using existing resource group 'lab-product-framework'
⚙️ Running: az deployment group create --name product-framework --resource-group lab-product-framework --template-file main.bicep --parameters params.json 
✅ Deployment 'product-framework' succeeded ⌚ 10:56:06.061391 :50s]


<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the required outputs from the Bicep deployment.

In [15]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    pricing_dcr_endpoint = utils.get_deployment_output(output, 'pricingDCREndpoint', 'Pricing DCR Endpoint')
    pricing_dcr_immutable_id = utils.get_deployment_output(output, 'pricingDCRImmutableId', 'Pricing DCR ImmutableId')
    pricing_dcr_stream = utils.get_deployment_output(output, 'pricingDCRStream', 'Pricing DCR Stream')
    subscription_quota_dcr_endpoint = utils.get_deployment_output(output, 'subscriptionQuotaDCREndpoint', 'Subscription Quota DCR Endpoint')
    subscription_quota_dcr_immutable_id = utils.get_deployment_output(output, 'subscriptionQuotaDCRImmutableId', 'Subscription Quota DCR ImmutableId')
    subscription_quota_dcr_stream = utils.get_deployment_output(output, 'subscriptionQuotaDCRStream', 'Subscription Quota DCR Stream')
    
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")


⚙️ Running: az deployment group show --name product-framework -g lab-product-framework 
✅ Retrieved deployment: product-framework ⌚ 10:57:00.680083 :3s]
👉🏽 APIM API Gateway URL: https://apim-dcsmdldxfng3y.azure-api.net
👉🏽 Pricing DCR Endpoint: https://dcr-pricing-dcsmdldxfng3y-msq2-westeurope.logs.z1.ingest.monitor.azure.com
👉🏽 Pricing DCR ImmutableId: dcr-4b622af71f70480c9d82deaedfa9b2bc
👉🏽 Pricing DCR Stream: Custom-Json-PRICING_CL
👉🏽 Subscription Quota DCR Endpoint: https://dcr-quota-dcsmdldxfng3y-2gnp-westeurope.logs.z1.ingest.monitor.azure.com
👉🏽 Subscription Quota DCR ImmutableId: dcr-0b2af312645b4c27ada67f32b8bfdc30
👉🏽 Subscription Quota DCR Stream: Custom-Json-SUBSCRIPTION_QUOTA_CL
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****657d
👉🏽 Subscription Name: subscription2
👉🏽 Subscription Key: ****d1e4
👉🏽 Subscription Name: subscription3
👉🏽 Subscription Key: ****1c40


<a id='pricing'></a>
### 🔍 Display retail pricing info based on the [pricing API](https://learn.microsoft.com/en-us/rest/api/cost-management/retail-prices/azure-retail-prices)



In [16]:
import requests
from tabulate import tabulate 

def build_pricing_table(json_data, table_data):
    for item in json_data['Items']:
        meter = item['meterName']
        table_data.append([item['armRegionName'], item['armSkuName'], item['retailPrice']*1000])

table_data = []
table_data.append(['Region', 'SKU', 'Retail Price'])
for aiservice in aiservices_config:
    aiservice_resource_location = aiservice['location']    
    prices = requests.get(f"https://prices.azure.com/api/retail/prices?currencyCode='{currency_code}'&$filter=serviceName eq 'Foundry Models' and unitOfMeasure eq '1K' and armRegionName eq '{aiservice_resource_location}'")
    if prices.status_code == 200:
        prices_json = prices.json()
        build_pricing_table(prices_json, table_data)
    print(tabulate(table_data, headers='firstrow', tablefmt='psql'))


+---------------+---------------------------------------------+----------------+
| Region        | SKU                                         |   Retail Price |
|---------------+---------------------------------------------+----------------|
| swedencentral | o3 mini 0131 Batch Outp Data Zone           |            1.8 |
| swedencentral | gpt 4.1 nano cached Inp glbl                |            0   |
| swedencentral | gpt4omini-rt-aud1217 Outp regnl             |           17.5 |
| swedencentral | gpt-4o-aud-0603-txt Inp DZone               |            2   |
| swedencentral | gpt-4o-rt-aud-0603 Outp glbl                |           58   |
| swedencentral | Phi-4-reasoning-Output                      |            0.4 |
| swedencentral | o1-pro Inp regnl                            |          131.6 |
| swedencentral | o3 0416 Batch Outp glbl                     |            2.9 |
| swedencentral | gpt 4.1 Inp regnl                           |            1.8 |
| swedencentral | gpt-4o-rt-

<a id='4'></a>
### 4️⃣ Load the pricing data into Azure Monitor custom table

👉 This script uses retail price information. Please adjust it to apply a discount or to use a flat rate with PTUs.   
👉 We are multiplying by 1000 to get the retail price per 1K tokens.   
👉 Deploy this script as a [job](https://learn.microsoft.com/en-us/azure/container-apps/jobs?tabs=azure-cli) to run automatically on a predefined schedule.

In [17]:
import requests
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=pricing_dcr_endpoint, credential=credential, logging_enable=False)

for aiservice in aiservices_config:
    aiservice_resource_location = aiservice['location']
    prices = requests.get(f"https://prices.azure.com/api/retail/prices?currencyCode='{currency_code}'&$filter=serviceName eq 'Foundry Models' and unitOfMeasure eq '1K' and armRegionName eq '{aiservice_resource_location}'")
    if prices.status_code == 200:
        prices_json = prices.json()
        if prices_json and 'Items' in prices_json:
            for deployment in models_config:
                input_tokens_price = next((item['retailPrice'] * 1000 for item in prices_json['Items'] if item.get('skuName') == deployment.get("inputTokensMeterSku")), None)
                output_tokens_price = next((item['retailPrice'] * 1000 for item in prices_json['Items'] if item.get('skuName') == deployment.get("outputTokensMeterSku")), None)
                utils.print_info(f"Adding model {deployment.get("name")} with input / output tokens price {input_tokens_price} / {output_tokens_price}")
                body = [{ "TimeGenerated": str(datetime.now(timezone.utc)),
                        "Model": deployment.get("name"),
                        "InputTokensPrice": input_tokens_price,
                        "OutputTokensPrice": output_tokens_price }]
                try:
                    client.upload(rule_id=pricing_dcr_immutable_id, stream_name=pricing_dcr_stream, logs=body)
                    utils.print_ok(f"Upload succeeded for model {deployment.get("name")}")
                except HttpResponseError as e:
                    utils.print_error(f"Upload failed: {e}")            


👉🏽 Adding model gpt-5-mini with input / output tokens price None / None
✅ Upload succeeded for model gpt-5-mini ⌚ 10:57:16.400454 
👉🏽 Adding model gpt-5 with input / output tokens price None / None
✅ Upload succeeded for model gpt-5 ⌚ 10:57:16.557874 


<a id='5'></a>
### 5️⃣ Load the Subscription Quota into Azure Monitor custom table


In [18]:
import requests
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=subscription_quota_dcr_endpoint, credential=credential, logging_enable=False)

for subscription in apim_subscriptions_config:
    for product in apim_products_config:
        if product.get("name") == subscription.get("product"):
            cost_quota = product.get("costQuota")
            utils.print_info(f"Adding {subscription.get('name')} with cost quota {cost_quota}")
            body = [{ 
                "TimeGenerated": str(datetime.now(timezone.utc)),
                "Subscription": subscription.get("name"),
                "Email": subscription.get("email"),
                "CostQuota": cost_quota
            }]
            try:
                client.upload(rule_id=subscription_quota_dcr_immutable_id, stream_name=subscription_quota_dcr_stream, logs=body)
                utils.print_ok(f"Upload succeeded for {subscription.get("name")}")
            except HttpResponseError as e:
                utils.print_error(f"Upload failed: {e}")            


👉🏽 Adding subscription1 with cost quota 100
✅ Upload succeeded for subscription1 ⌚ 10:57:23.885748 
👉🏽 Adding subscription2 with cost quota 100
✅ Upload succeeded for subscription2 ⌚ 10:57:24.059112 
👉🏽 Adding subscription3 with cost quota 100
✅ Upload succeeded for subscription3 ⌚ 10:57:24.436749 


<a id='sdk'></a>
### 🧪 Execute multiple runs using the Azure OpenAI Python SDK

👉 We will send requests with random subscriptions while respecting product-to-model mapping (`gpt5 -> gpt-5`, `gpt5mini -> gpt-5-mini`, `unlimited -> any`). Adjust the `sleep_time_ms` and the number of `runs` to your test scenario.


In [19]:
import time, random
from openai import AzureOpenAI

runs = 10
sleep_time_ms = 100

for i in range(runs):
    apim_subscription = random.choice(apim_subscriptions)
    subscription_product = next((item.get("product") for item in apim_subscriptions_config if item.get("name") == apim_subscription.get("name")), None)

    if subscription_product == "gpt5":
        openai_model = next(model for model in models_config if model.get("name") == "gpt-5")
    elif subscription_product == "gpt5mini":
        openai_model = next(model for model in models_config if model.get("name") == "gpt-5-mini")
    else:
        openai_model = random.choice(models_config)
    client = AzureOpenAI(
        azure_endpoint = f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key = apim_subscription.get("key"),
        api_version = inference_api_version
    )
    try:
        response = client.chat.completions.create(
            model = str(openai_model.get('name')),
            messages = [
                {"role": "user", "content": "Can you tell me the time, please?"}
            ],
            extra_headers = {"x-user-id": "alex"}
        )
        print(f"▶️ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] 💬 {response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] Error: {e}")
    time.sleep(sleep_time_ms/1000)


▶️ Run 1/10: [subscription2 w/ gpt-5-mini] 💬 I don’t have access to a live clock, so I can’t read the current time from your device. I can help in other ways though — which would you prefer?

- Tell me your city or time zone and I’ll tell you the current time there (you may need to confirm your device’s clock if you want absolute accuracy).  
- Tell me two places and I’ll convert between their times or compute the time difference.  
- Want quick ways to check on your device? Try: “Hey Siri / OK Google / Alexa, what time is it?”, look at your phone or computer corner/lock screen, or run the command-line date/time command (e.g., date on macOS/Linux or wmic path win32_localtime get /format:list on Windows).

Which option would you like?
▶️ Run 2/10: [subscription1 w/ gpt-5] 💬 Sorry—I don’t have access to the current time. Please check your device’s clock (status bar, lock screen, or menu bar). If you tell me your city or time zone, I can help convert that time to other locations or set up

<a id='workbooks'></a>
### 🔍 Open the dashboard and workbooks in the Azure Portal

👉 The Cost Analysis workbook contains information on the total costs and quotas for each subscription.  
👉 The [Azure OpenAI Insights workbook](https://github.com/dolevshor/Azure-OpenAI-Insights) provides comprehensive details about service and model usage. Credits to [Dolev Shor](https://github.com/dolevshor/Azure-OpenAI-Insights).  
👉 The [Alerts workbook](https://github.com/microsoft/AzureMonitorCommunity/tree/master/Azure%20Services) provides information about the alerts triggered by Azure Monitor.  

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.